In [1]:
import numpy as np
data = np.load('../data/processed/mafaulda_windows.npz', allow_pickle=True)
X, y, loc_tags = data['X'], data['y'], data['loc_tags']
train_idx, test_idx = data['train_idx'].tolist(), data['test_idx'].tolist()


In [3]:
import pandas as pd
from scipy.stats import kurtosis
from scipy.fft import fft, fftfreq

def extract(w, fs = 50000):
    rms = np.sqrt(np.mean(w**2))
    kurt = kurtosis(w)
    crest = np.max(np.abs(w)) / rms
    n = len(w)
    freq = fftfreq(n, 1/fs)[:n//2]
    mag = np.abs(fft(w))[:n//2] * 2/n
    edges = np.linspace(0, fs/2, 6)
    bands = {}
    for i in range(5):
        mask = (freq >= edges[i]) & (freq < edges[i+1])
        bands[f"band_{i}"] = np.sum(mag[mask]**2)
    return {"rms": rms, "kurtosis": kurt, "crest": crest, **bands}

feat_df = pd.DataFrame([extract(w) for w in X])
feat_df["label"] = y
feat_df["location"] = loc_tags
feat = [c for c in feat_df.columns if c not in ("label", "location")]
print(feat_df.shape)

(141449, 10)


In [4]:
log_feats = [ f for f in feat if f != "kurtosis"]
feat_df_log = feat_df.copy()
for f in log_feats:
    feat_df_log[f] = np.log1p(feat_df_log[f])

In [5]:
from sklearn.preprocessing import StandardScaler
from scipy.linalg import fractional_matrix_power

def coral_align(Xs, Xt, eps=1e-5):
    Cs = np.cov(Xs, rowvar=False) + eps * np.eye(Xs.shape[1])
    Ct = np.cov(Xt, rowvar=False) + eps * np.eye(Xt.shape[1])
    Cs_inv_sqrt = fractional_matrix_power(Cs, -0.5).real
    Ct_sqrt = fractional_matrix_power(Ct, 0.5).real
    return Xs @ Cs_inv_sqrt @ Ct_sqrt

train_scaler = StandardScaler().fit(feat_df_log.loc[train_idx, feat])
test_scaler = StandardScaler().fit(feat_df_log.loc[test_idx, feat])

Xtr_std = train_scaler.transform(feat_df_log.loc[train_idx, feat])
Xte_std = test_scaler.transform(feat_df_log.loc[test_idx, feat])
Xtr_coral = coral_align(Xtr_std, Xte_std)

ytr = feat_df_log.loc[train_idx, "label"].values
yte = feat_df_log.loc[test_idx, "label"].values


In [6]:
from sklearn.ensemble import RandomForestClassifier

ytr_stage1 = (ytr == "ball_fault").astype(int)
yte_stage1 = (yte == "ball_fault").astype(int)

gate = RandomForestClassifier(n_estimators=200, random_state=0)
gate.fit(Xtr_coral, ytr_stage1)

gate_preds = gate.predict(Xte_std)
print("Stage 1 gate accuracy (ball_fault vs rest):", (gate_preds == yte_stage1).mean())


Stage 1 gate accuracy (ball_fault vs rest): 0.9460015881885827


In [7]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(yte_stage1, gate_preds, target_names=["not_ball_fault", "ball_fault"]))
print(confusion_matrix(yte_stage1, gate_preds))


                precision    recall  f1-score   support

not_ball_fault       0.94      0.99      0.97     51425
    ball_fault       0.96      0.81      0.88     16577

      accuracy                           0.95     68002
     macro avg       0.95      0.90      0.92     68002
  weighted avg       0.95      0.95      0.94     68002

[[50884   541]
 [ 3131 13446]]


In [8]:
stage2_train_mask = ytr != "ball_fault"
stage2_test_mask = yte != "ball_fault"

Xtr_stage2 = Xtr_coral[stage2_train_mask]
ytr_stage2 = ytr[stage2_train_mask]
Xte_stage2 = Xte_std[stage2_test_mask]
yte_stage2 = yte[stage2_test_mask]

stage2_clf = RandomForestClassifier(n_estimators=200, random_state=0)
stage2_clf.fit(Xtr_stage2, ytr_stage2)
print("Stage 2 accuracy, ASSUMING PERFECT stage-1 routing:", stage2_clf.score(Xte_stage2, yte_stage2))


Stage 2 accuracy, ASSUMING PERFECT stage-1 routing: 0.40274185707340787


In [9]:
preds_stage2 = stage2_clf.predict(Xte_stage2)
print(classification_report(yte_stage2, preds_stage2))


              precision    recall  f1-score   support

  cage_fault       0.37      0.42      0.39     22748
      normal       0.00      0.00      0.00      5929
  outer_race       0.44      0.49      0.46     22748

    accuracy                           0.40     51425
   macro avg       0.27      0.30      0.28     51425
weighted avg       0.36      0.40      0.38     51425



/Users/yashralhan/Projects/bearing-fault-detection/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/yashralhan/Projects/bearing-fault-detection/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/yashralhan/Projects/bearing-fault-detection/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to co

In [10]:
final_preds = np.empty(len(yte), dtype=object)

is_ball_fault_pred = gate.predict(Xte_std) == 1
final_preds[is_ball_fault_pred] = "ball_fault"

not_ball_idx = np.where(~is_ball_fault_pred)[0]
final_preds[not_ball_idx] = stage2_clf.predict(Xte_std[not_ball_idx])

print("Full cascade accuracy:", (final_preds == yte).mean())


Full cascade accuracy: 0.5005735125437487


In [11]:
print(classification_report(yte, final_preds))
labels_order = sorted(set(yte))
cm_cascade = confusion_matrix(yte, final_preds, labels=labels_order)
pd.DataFrame(cm_cascade, index=labels_order, columns=labels_order)


/Users/yashralhan/Projects/bearing-fault-detection/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/yashralhan/Projects/bearing-fault-detection/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

  ball_fault       0.96      0.81      0.88     16577
  cage_fault       0.34      0.42      0.37     22748
      normal       0.00      0.00      0.00      5929
  outer_race       0.42      0.49      0.45     22748

    accuracy                           0.50     68002
   macro avg       0.43      0.43      0.43     68002
weighted avg       0.49      0.50      0.49     68002



/Users/yashralhan/Projects/bearing-fault-detection/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,ball_fault,cage_fault,normal,outer_race
ball_fault,13446,1972,0,1159
cage_fault,541,9444,0,12763
normal,0,4680,0,1249
outer_race,0,11598,0,11150


In [12]:
results_cascade = {
    "log+CORAL (flat 4-way)": 0.5193,
    "severity-first cascade": (final_preds == yte).mean(),
}
pd.Series(results_cascade).sort_values(ascending=False)


log+CORAL (flat 4-way)    0.519300
severity-first cascade    0.500574
dtype: float64